In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")

In [4]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [5]:
df['question'], df['answer'] = df['question'].str.lower(), df['answer'].str.lower()

In [6]:
import re

In [7]:
df[['question', 'answer']] = df[['question', 'answer']].apply(lambda col: col.str.replace(r'<.*?>', '', regex=True))

In [8]:
url_pattern = r'http[s]?://\S+|www\.\S+'

df['question'] = df['question'].str.replace(url_pattern, '', regex=True)
df['answer'] = df['answer'].str.replace(url_pattern, '', regex=True)

In [9]:
import string

punct_pattern = f'[{string.punctuation}]'

df['question'] = df['question'].str.replace(punct_pattern, ' ', regex=True)
# df['answer'] = df['answer'].str.replace(punct_pattern, ' ', regex=True)

In [10]:
pip install pyspellchecker


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
from spellchecker import SpellChecker

spell = SpellChecker()

misspelled_words = set()

def find_misspelled(text):
  words = text.split()
  unknown = spell.unknown(words)
  misspelled_words.update(unknown)

In [12]:
df['question'].apply(find_misspelled)
# df['answer'].apply(find_misspelled)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Name: question, Length: 90, dtype: object

In [13]:
misspelled_words

{'ii', 'tv', 'uk'}

In [ ]:
# pip install emoji


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import emoji

emoji_list = []

def find_emojis(text):
    found = emoji.emoji_list(text)
    for item in found:
        emoji_list.append(item['emoji'])

df['question'].apply(find_emojis)
df['answer'].apply(find_emojis)

print(emoji_list)

[]


In [16]:
# import emoji

# def handle_emojis(text):
#     return emoji.demojize(text)

# df['question'] = df['question'].apply(handle_emojis)
# df['answer'] = df['answer'].apply(handle_emojis)

In [17]:
df.head()

,question,answer
0,what is the capital of france,paris
1,what is the capital of germany,berlin
2,who wrote to kill a mockingbird,harper-lee
3,what is the largest planet in our solar system,jupiter
4,what is the boiling point of water in celsius,100


In [ ]:
import nltk
# nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/nayan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [19]:
nltk.word_tokenize(df['question'][0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [20]:
# vocab
vocab = {'<UNK>':0}

In [21]:
def build_vocab(row):
  tokenized_question = nltk.word_tokenize(row['question'])
  tokenized_answer = nltk.word_tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [22]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [23]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [24]:
len(vocab)

324

In [25]:
# convert words to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []

  text = text.lower()
  text = re.sub(punct_pattern, ' ', text)

  for token in nltk.word_tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [26]:
text_to_indices("What is campusx?", vocab)

[1, 2, 0]

In [27]:
import torch
from torch.utils.data import Dataset, DataLoader

In [28]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_questions = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answers = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_questions), torch.tensor(numerical_answers)

In [29]:
dataset = QADataset(df, vocab)

In [30]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [31]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[10,  2,  3, 66,  5, 67]]) tensor([0, 0])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([36])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([179])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([155])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([307])
tensor([[ 10,  11, 189, 158, 190]]) tensor([191])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([170])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([136])
tensor([[  1,   2,   3,  50, 180, 181, 182, 183]]) tensor([184])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([41])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([95])
tensor([[  1,  87, 229, 230, 231, 232]]) tensor([233])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([116])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([238])
tensor([[  1,   2,   3,   4,   5,

In [32]:
import torch.nn as nn

In [33]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [34]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [35]:
embed = nn.Embedding(324, embedding_dim=50)

In [36]:
a = embed(dataset[0][0])

In [37]:
a.shape

torch.Size([6, 50])

In [38]:
rnn = nn.RNN(50, 64)

In [39]:
b = rnn(a)

In [40]:
b[0].shape

torch.Size([6, 64])

In [41]:
b[1].shape

torch.Size([1, 64])

In [42]:
fc = nn.Linear(64, 324)

In [43]:
c = fc(b[1])

In [44]:
c.shape

torch.Size([1, 324])

In [45]:
# mlflow imports
import mlflow
import mlflow.pytorch

In [46]:
mlflow.set_experiment("next_word_predictor")

2026/08/07 02:11:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/07 02:11:08 INFO mlflow.store.db.utils: Updating database tables
2026/08/07 02:11:09 INFO mlflow.tracking.fluent: Experiment with name 'next_word_predictor' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/nayan/next_word_predictor/mlruns/1', creation_time=1786047969201, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786047969201, lifecycle_stage='active', name='next_word_predictor', tags={}, trace_location=None, workspace='default'>

In [47]:
learning_rate = 0.001
epochs = 20

In [48]:
model = SimpleRNN(len(vocab))

In [49]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [55]:
# mlflow start run
with mlflow.start_run():

  # log hyperparameters
  mlflow.log_param("learning_rate", learning_rate)
  mlflow.log_param("epochs", epochs)
  mlflow.log_param("embedding_dim", 50)
  mlflow.log_param("hidden_size", 64)
  mlflow.log_param("vocab_size", len(vocab))

  # training loop
  for epoch in range(epochs):

    total_loss = 0
    correct = 0
    total_loss = 0

    for question, answer in dataloader:

      optimizer.zero_grad()

      # forward pass
      output = model(question)
      target = answer[0][0].unsqueeze(0)

      # loss -> output shape (1,324) - (1)
      loss = criterion(output, target)

      # gradients
      loss.backward()

      # update parameters
      optimizer.step()

      total_loss = total_loss + loss.item()

      # accuracy check
      predicted = torch.argmax(output, dim=1)
      correct += (predicted == target).sum().item()
      total += target.size(0)


    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total  

    print(f"Epoch: {epoch+1}, Avg Loss: {avg_loss}, Accuracy: {accuracy:.2%}")

    # log loss for this epoch, so MLflow shows a loss-over-time chart
    mlflow.log_metric("loss", avg_loss, step=epoch)
    mlflow.log_metric("accuracy", accuracy, step=epoch)

  # log the final loss as a top-level metric too
  mlflow.log_metric("final_loss", avg_loss)
  mlflow.log_metric("final_accuracy", accuracy)

  # log the trained model itself, versioned under this run
  mlflow.pytorch.log_model(
    model,
    "model",
    registered_model_name="next_word_predictor_model",
    serialization_format="pickle",
  )

  print(f"MLflow run ID: {mlflow.active_run().info.run_id}")

NameError: name 'total' is not defined

In [56]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [57]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [58]:
# Run this in your notebook AFTER training, to save what the API needs

import json
import torch

torch.save(model.state_dict(), "model.pth")

with open("vocab.json", "w") as f:
    json.dump(vocab, f)

print("Saved model.pth and vocab.json")

Saved model.pth and vocab.json
